# 03_chargement_duckdb

# Chargement DuckDB

Ce notebook a servi à explorer et prototyper le chargement des données
nettoyées (poi, connections) dans DuckDB, avec upsert et contraintes de
clé étrangère.

La logique a depuis été migrée vers `src/warehouse/duckdb_loader.py`,
avec tests unitaires associés (`tests/test_warehouse_duckdb_loader.py`).

Ce notebook sert maintenant d'exemple d'usage du module.

In [7]:
import logging
from pathlib import Path

import duckdb
import polars as pl

from warehouse.duckdb_loader import charger_openchargemap_dans_duckdb

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")

ROOT_PATH = Path.cwd().resolve().parent

# Chargement des DataFrames depuis les fichiers Parquet (issus du notebook 02)
poi_df = pl.read_parquet(ROOT_PATH / "data" / "processed" / "2026-08-13_110043_poi.parquet")
connections_df = pl.read_parquet(ROOT_PATH / "data" / "processed" / "2026-08-13_110043_connections.parquet")

# Connexion à la base DuckDB
con = duckdb.connect(str(ROOT_PATH / "data" / "warehouse" / "electric_mobility.duckdb"))

charger_openchargemap_dans_duckdb(con, poi_df, connections_df)

2026-08-13 11:10:22,556 - warehouse.duckdb_loader - INFO - Table POI créée avec succès.
2026-08-13 11:10:22,579 - warehouse.duckdb_loader - INFO - Insertion données dans la table POI : 50 lignes traitées avec succès
2026-08-13 11:10:22,585 - warehouse.duckdb_loader - INFO - Table connections créée avec succès.
2026-08-13 11:10:22,606 - warehouse.duckdb_loader - INFO - Insertion données dans la table Connections : 99 lignes traitées avec succès
2026-08-13 11:10:22,607 - warehouse.duckdb_loader - INFO - Chargement DuckDB terminé : poi=(50, 10), connections=(99, 10)


In [8]:
print(poi_df.filter(pl.col("title").str.contains("SAEMES")))
print(connections_df.filter(pl.col("connection_id") == 331827))

shape: (1, 10)
┌────────┬─────────────┬──────┬──────────┬───┬─────────────┬────────────┬─────────────┬────────────┐
│ poi_id ┆ title       ┆ town ┆ postcode ┆ … ┆ number_of_p ┆ usage_cost ┆ date_last_c ┆ town_norma │
│ ---    ┆ ---         ┆ ---  ┆ ---      ┆   ┆ oints       ┆ ---        ┆ onfirmed    ┆ lisee      │
│ i64    ┆ str         ┆ str  ┆ str      ┆   ┆ ---         ┆ str        ┆ ---         ┆ ---        │
│        ┆             ┆      ┆          ┆   ┆ i64         ┆            ┆ str         ┆ str        │
╞════════╪═════════════╪══════╪══════════╪═══╪═════════════╪════════════╪═════════════╪════════════╡
│ 198782 ┆ SAEMES |    ┆ null ┆ null     ┆ … ┆ null        ┆ null       ┆ null        ┆ null       │
│        ┆ PARKING     ┆      ┆          ┆   ┆             ┆            ┆             ┆            │
│        ┆ LAGRANGE    ┆      ┆          ┆   ┆             ┆            ┆             ┆            │
└────────┴─────────────┴──────┴──────────┴───┴─────────────┴────────────┴───

In [9]:
con.close()